# Gaussian HSMM for yield-curve regimes

This notebook mirrors the HMM analysis in `main.ipynb`, but uses the published [`gaussian-hsmm`](https://pypi.org/project/gaussian-hsmm/) package to model regime durations explicitly.

Install the published release once in this notebook's Python environment:

```powershell
python -m pip install gaussian-hsmm==0.1.0
```

The PyPI distribution name contains a hyphen (`gaussian-hsmm`), while Python imports it with an underscore (`gaussian_hsmm`). An HMM asks whether the process stays or switches at every date, which implies geometric spell lengths. An HSMM instead assigns probability directly to the complete duration of each state spell.

The package uses a shifted, truncated Poisson duration: $D-1\sim\operatorname{Poisson}(\lambda_j)$ in state $j$, normalized over `1..max_duration`. The shift guarantees that every spell lasts at least one observation; truncation prevents the algorithm from considering infinitely many possible spell lengths. Its transition matrix has a zero diagonal because persistence is represented by the duration distribution rather than self-transitions.

`GaussianHSMM.fit` first fits `hmmlearn.GaussianHMM` only to obtain reasonable starting parameters. It then performs explicit-duration forward–backward inference and EM-style HSMM updates. Forward–backward sums over possible hidden state/spell explanations; EM alternates between estimating their probabilities and updating parameters. All later calls to `score`, `predict`, `predict_proba`, and `decode` use the final HSMM.

A hidden state is not observed directly and its number has no built-in economic meaning. The model learns a probability distribution for the measurements within each state. We interpret a state only afterward by examining its means, uncertainty, spell duration, and dates.


In [ ]:
from pathlib import Path

import gaussian_hsmm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gaussian_hsmm import GaussianHSMM
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=4, suppress=True)

# These lines verify that the notebook is using the installed PyPI package,
# rather than an old class copied into this notebook or a manual sys.path entry.
print(f"gaussian-hsmm version: {gaussian_hsmm.__version__}")
print(f"Imported from: {Path(gaussian_hsmm.__file__).resolve()}")


## 1. Reconstruct the same information set

The model observes five features; the hidden states are recurring joint configurations of these features.

- Nelson–Siegel `level`, `slope`, and `curvature` summarize the yield curve.
- `BE5` is five-year breakeven inflation. [FRED defines it](https://fred.stlouisfed.org/series/T5YIE) as market inflation compensation derived from nominal and inflation-indexed Treasury yields.
- `policy_spread = 2Y Treasury yield - EFFR` compares the market two-year rate with the current overnight effective federal funds rate. It contains expected-rate-path information plus risk and term premia; it is not a pure Federal Reserve forecast.

Treasury documents its [constant-maturity yield construction](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve). The split remains chronological. The scaler is fitted only on training observations so later information cannot influence preprocessing.

Standardization replaces each feature $x$ by $(x-\text{training mean})/\text{training standard deviation}$. This prevents a feature with numerically larger units from dominating the Gaussian likelihood simply because of its scale. The original economic units are restored when state means are displayed later.


In [ ]:
# Locate data whether the kernel starts in the repository root or notebooks/.
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")

# Load the three Dynamic Nelson–Siegel factors. Renaming Beta 1/2/3 makes
# every later expression readable in economic rather than positional terms.
ns = pd.read_csv(DATA_DIR / "ns.csv").rename(
    columns={"Beta 1": "level", "Beta 2": "slope", "Beta 3": "curvature"}
)
# A DatetimeIndex lets pandas align separate sources by the actual date.
ns["Date"] = pd.to_datetime(ns["Date"])
ns = ns.set_index("Date")

# Load market-implied five-year inflation compensation.
be5 = pd.read_csv(DATA_DIR / "T5YIE.csv").rename(
    columns={"observation_date": "Date", "T5YIE": "BE5"}
)
be5["Date"] = pd.to_datetime(be5["Date"])
be5 = be5.set_index("Date")

# Load EFFR, the effective overnight federal funds rate.
effr = pd.read_csv(DATA_DIR / "EFFR.csv").rename(
    columns={"observation_date": "Date"}
)
effr["Date"] = pd.to_datetime(effr["Date"])
effr = effr.set_index("Date")

# Load the Federal Reserve H.15 curve and keep a short name for its 2-year
# maturity. The exact long header comes directly from the downloaded CSV.
maturities = pd.read_csv(DATA_DIR / "FRB_H15.csv").rename(
    columns={
        "Series Description": "Date",
        "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y",
    }
)
maturities["Date"] = pd.to_datetime(maturities["Date"])
maturities = maturities.set_index("Date")

# A left merge keeps every available 2-year date and aligns EFFR on date.
rates = maturities[["2Y"]].merge(
    effr[["EFFR"]], left_index=True, right_index=True, how="left"
)
# Non-numeric missing markers become NaN instead of causing arithmetic errors.
rates = rates.apply(pd.to_numeric, errors="coerce")
# Positive values mean 2Y is above today's overnight rate; negative values
# mean it is below. Expectations and risk premia can both affect this spread.
rates["policy_spread"] = rates["2Y"] - rates["EFFR"]

# This list fixes both the model's input columns and their order.
features = ["level", "slope", "curvature", "BE5", "policy_spread"]
# Merge on dates, convert all columns to numbers, remove incomplete rows, and
# sort chronologically. An HSMM assumes row order is time order.
hsmm_data = (
    ns.merge(be5[["BE5"]], left_index=True, right_index=True, how="left")
    .merge(rates[["policy_spread"]], left_index=True, right_index=True, how="left")
    [features]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
    .sort_index()
)

# A chronological 80/20 split is appropriate for time series; random shuffling
# would train on the future and evaluate on the past.
split = int(0.80 * len(hsmm_data))
train_df = hsmm_data.iloc[:split].copy()
test_df = hsmm_data.iloc[split:].copy()

# Learn means/standard deviations from training only. transform() reuses those
# exact values for later dates and therefore does not peek at their distribution.
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df)
X_test = scaler.transform(test_df)
# Recombine standardized blocks only for the later retrospective state plot.
X_all = np.vstack([X_train, X_test])

print(f"Train: {train_df.index.min().date()} to {train_df.index.max().date()} ({len(train_df):,})")
print(f"Test:  {test_df.index.min().date()} to {test_df.index.max().date()} ({len(test_df):,})")


## 2. Tune state count and covariance structure

The package exposes a scikit-learn-style estimator. Constructing `GaussianHSMM(...)` stores settings; `.fit(X_train)` estimates model parameters.

- **Log-likelihood:** likelihood multiplies many small conditional probabilities; log-likelihood turns that product into a numerically stable sum. Values are often negative, so `-10,000` is larger/better than `-12,000`. Larger means the observed sequence is more plausible under the model, but training likelihood generally improves whenever flexibility is added.
- **AIC:** $2k-2\log L$. Lower is preferred; its complexity penalty is relatively light.
- **BIC:** $k\log(n)-2\log L$. Lower is preferred; it penalizes extra parameters more strongly in a long sample.
- **Held-out log-likelihood per observation:** higher means the model trained on earlier dates better explains later dates.

Diagonal covariance estimates one variance per feature and state and treats residual feature movements as conditionally uncorrelated inside a state. Full covariance additionally estimates every within-state covariance/correlation. The latter can represent tilted ellipsoidal clusters but uses many more parameters and is easier to overfit or make numerically singular.

`max_duration=126` represents spell lengths from one through roughly half a trading year. If a fitted duration mean approaches the cap, raise the cap and refit. Tuning uses only two duration-aware iterations to limit runtime; the final specification is refitted more thoroughly.


In [ ]:
STATE_COUNTS = range(2, 5)
COVARIANCE_TYPES = ("diag", "full")
MAX_DURATION = 126
TUNING_HSMM_ITERATIONS = 2

candidate_models = {}
tuning_rows = []

for covariance_type in COVARIANCE_TYPES:
    for n_states in STATE_COUNTS:
        candidate = GaussianHSMM(
            # Number of latent regimes to estimate.
            n_components=n_states,
            # `diag` estimates feature variances; `full` also estimates correlations.
            covariance_type=covariance_type,
            # Largest explicitly represented state duration, in observations.
            max_duration=MAX_DURATION,
            # Duration-aware HSMM EM iterations after HMM initialization.
            n_iter=TUNING_HSMM_ITERATIONS,
            # Iteration limit for the initializer, not the final HSMM.
            hmm_n_iter=300,
            # Stop when consecutive HSMM likelihoods differ by less than 0.001.
            tol=1e-3,
            # Covariance floor that helps prevent singular Gaussian estimates.
            min_covar=1e-4,
            # Reproducible HMM initialization.
            random_state=42,
        ).fit(X_train)

        candidate_models[(n_states, covariance_type)] = candidate

        # `score` uses the explicit-duration forward algorithm and integrates
        # over all valid state segmentations; it is not a Viterbi path score.
        train_log_likelihood = candidate.score(X_train)
        test_log_likelihood = candidate.score(X_test)

        tuning_rows.append(
            {
                "n_states": n_states,
                "covariance_type": covariance_type,
                "train_log_likelihood": train_log_likelihood,
                "test_log_likelihood_per_observation": test_log_likelihood / len(X_test),
                # These package methods use the fitted free-parameter count.
                "aic": candidate.aic(X_train),
                "bic": candidate.bic(X_train),
                "parameters": candidate.n_parameters_,
                "converged": candidate.monitor_.converged,
            }
        )

hsmm_tuning = pd.DataFrame(tuning_rows)
display(hsmm_tuning.sort_values("bic").reset_index(drop=True))

print("Lowest BIC candidate:")
display(hsmm_tuning.loc[[hsmm_tuning["bic"].idxmin()]])
print("Best held-out likelihood candidate:")
display(
    hsmm_tuning.loc[
        [hsmm_tuning["test_log_likelihood_per_observation"].idxmax()]
    ]
)


## 3. Refit and interpret the three-state diagonal HSMM

The grid is a sensitivity analysis. This section retains the requested three-state diagonal specification so it remains comparable with `main.ipynb`.

`transmat_[i,j]` is the probability of entering state $j$ **after the entire spell in state $i$ ends**. The zero diagonal does not mean the model switches every day: `duration_means_` and `duration_probs_` control how long it remains in the current state.

`predict` returns one duration-aware Viterbi path: the single joint state sequence with the highest probability. `predict_proba` instead sums over all possible paths and returns smoothed state probabilities, preserving uncertainty. A date can therefore be 60% state 0 and 40% state 1 even though the Viterbi output must choose one label. Because smoothing uses the complete sequence supplied, these probabilities are appropriate for retrospective interpretation—not as real-time forecasting features without a walk-forward construction.


In [ ]:
model = GaussianHSMM(
    n_components=3,
    covariance_type="diag",
    max_duration=MAX_DURATION,
    # Use more duration-aware updates for the final research specification.
    n_iter=6,
    hmm_n_iter=500,
    tol=1e-3,
    min_covar=1e-4,
    random_state=42,
    # Print the explicit-duration log-likelihood after each update.
    verbose=True,
).fit(X_train)

print("Initializer type:", type(model.hmm_initializer_).__name__)
print("Final estimator type:", type(model).__name__)

# Retrospective inference uses the whole standardized sequence. The fitted
# parameters come only from X_train, but smoothing within X_all can use later
# observations when assigning an earlier date's posterior probability.
states = model.predict(X_all)
probabilities = model.predict_proba(X_all)

hsmm_results = hsmm_data.copy()
hsmm_results["state"] = states
for state in range(model.n_components):
    hsmm_results[f"state_{state}_probability"] = probabilities[:, state]
hsmm_results["classification_confidence"] = probabilities.max(axis=1)

# Emission means were estimated in standardized units. Convert them back so
# each state can be interpreted in the original economic units.
state_means = pd.DataFrame(
    scaler.inverse_transform(model.means_), columns=features
)
state_means.index.name = "state"

if model.covariance_type == "diag":
    state_std = pd.DataFrame(
        np.sqrt(model.covars_) * scaler.scale_, columns=features
    )
else:
    scale_matrix = np.diag(scaler.scale_)
    original_covariances = np.asarray(
        [scale_matrix @ covariance @ scale_matrix for covariance in model.covars_]
    )
    state_std = pd.DataFrame(
        np.sqrt(np.diagonal(original_covariances, axis1=1, axis2=2)),
        columns=features,
    )
state_std.index.name = "state"

transition_matrix = pd.DataFrame(
    model.transmat_,
    index=[f"from_state_{i}" for i in range(model.n_components)],
    columns=[f"to_state_{i}" for i in range(model.n_components)],
)
duration_table = pd.DataFrame(
    {
        "shifted_poisson_mean_trading_days": model.duration_means_,
        "duration_cap": model.max_duration,
    }
)
duration_table.index.name = "state"

print("State means in original units:")
display(state_means.round(3))
print("Within-state standard deviations:")
display(state_std.round(3))
print("Transition probabilities after a complete spell ends:")
display(transition_matrix.round(4))
print("Explicit duration estimates:")
display(duration_table.round(1))
print("HSMM convergence history:")
display(pd.Series(model.monitor_.history, name="train_log_likelihood").to_frame())

if np.any(model.duration_means_ > 0.75 * model.max_duration):
    print("WARNING: a duration mean is near max_duration; raise the cap and refit.")


## 4. Compare states with official policy dates

These dates come from official Federal Reserve records and are applied only after fitting:

- [December 16, 2008: target range established at 0–¼%](https://www.federalreserve.gov/newsevents/pressreleases/monetary20081216b.htm)
- [March 16, 2020: pandemic return to 0–¼%](https://www.federalreserve.gov/newsevents/pressreleases/monetary20200315a1.htm)
- [March 16, 2022: recent tightening cycle begins](https://www.federalreserve.gov/newsevents/pressreleases/monetary20220316a.htm)
- [July 26, 2023: target raised to 5¼–5½%](https://www.federalreserve.gov/publications/2023-ar-record-of-policy-actions-of-the-board-of-governors.htm)

These are interpretation anchors, not labels or causal validation. State numbers are arbitrary and may permute across fits. Describe states by their estimated economic profiles before assigning names.


In [ ]:
official_events = pd.DataFrame(
    [
        ("2008-12-16", "FOMC established 0 to 1/4 percent range"),
        ("2020-03-16", "pandemic return to 0 to 1/4 percent range"),
        ("2022-03-16", "recent tightening cycle begins"),
        ("2023-07-26", "target reaches 5-1/4 to 5-1/2 percent"),
    ],
    columns=["date", "official_event"],
)
official_events["date"] = pd.to_datetime(official_events["date"])


def state_on_or_before(date):
    """Return the last decoded state available on or before an event date."""
    available = hsmm_results.loc[hsmm_results.index <= date]
    return np.nan if available.empty else int(available.iloc[-1]["state"])


def summarize_spells(index, path):
    """Convert a date-level state path into one row per consecutive spell."""
    rows = []
    start = 0
    for end in range(1, len(path) + 1):
        if end == len(path) or path[end] != path[start]:
            rows.append(
                {
                    "state": int(path[start]),
                    "start": index[start],
                    "end": index[end - 1],
                    "trading_days": end - start,
                }
            )
            start = end
    return pd.DataFrame(rows)


official_events["decoded_state_on_or_before"] = official_events["date"].apply(
    state_on_or_before
)
display(official_events)

spells = summarize_spells(hsmm_results.index, states)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].step(hsmm_results.index, hsmm_results["state"], where="post")
axes[0].set_title("Published GaussianHSMM: retrospectively decoded regime")
axes[0].set_ylabel("state")
axes[0].set_yticks(range(model.n_components))

for state in range(model.n_components):
    axes[1].plot(
        hsmm_results.index,
        hsmm_results[f"state_{state}_probability"],
        label=f"state {state}",
        linewidth=1,
    )
axes[1].axvline(
    test_df.index[0], color="black", linestyle="--", alpha=0.6, label="train/test split"
)
axes[1].set_ylabel("smoothed probability")
axes[1].set_xlabel("date")
axes[1].legend(ncol=4)
plt.tight_layout()
plt.show()

print("Longest decoded spells:")
display(spells.sort_values("trading_days", ascending=False).head(15))

# Interpretation checklist:
# 1. Compare state means and use relative descriptions before assigning names.
# 2. Inspect probabilities; a hard Viterbi path hides classification uncertainty.
# 3. Compare fitted duration means with empirical decoded spell lengths.
# 4. Raise max_duration if a fitted mean approaches the cap.
# 5. Repeat random seeds and require stable economic profiles, not label numbers.
# 6. Use held-out likelihood as well as AIC/BIC when comparing specifications.
